# BTIS3043 Artificial Intelligence — Final Assessment (2026B)
**Core task:** Query all three eBook catalogues (Dataset A, B, C) using both predicate-only and fuzzy-enhanced reasoning, for two fixed scenarios, and compare the outcomes.

This notebook is the reproducible implementation backing the technical report. It imports the reusable engine modules from `src/`:
- `src/data_loader.py` — loads the three datasets
- `src/predicate_engine.py` — crisp/Boolean predicate querying (dataset-aware field search)
- `src/fuzzy_engine.py` — fuzzy membership functions + weighted aggregation (evidence-aware)


In [1]:
import sys, os
sys.path.insert(0, '.')
import pandas as pd
pd.set_option('display.max_colwidth', 90)

from src.data_loader import load_datasets
from src.predicate_engine import predicate_query, classify_relevance
from src.fuzzy_engine import (
    relevance_membership, recency_membership, affordability_membership,
    aggregate_fuzzy_score, build_affordability_thresholds
)

df_a, df_b, df_c = load_datasets('data')
DFS = {"A": df_a, "B": df_b, "C": df_c}
print("Dataset A (Existing Collection / Current Subscription):", df_a.shape)
print("Dataset B (Academic Catalogue):                        ", df_b.shape)
print("Dataset C (Acquisition / Licensing Catalogue):          ", df_c.shape)


Dataset A (Existing Collection / Current Subscription): (9, 11)
Dataset B (Academic Catalogue):                         (1743, 13)
Dataset C (Acquisition / Licensing Catalogue):           (807, 30)


## 1. Dataset and Knowledge Representation

| Dataset | Role | Searchable fields (predicate) | Price evidence | Discipline evidence |
|---|---|---|---|---|
| A — Existing eBook Collection | The DCS department's **current** holdings (9 records) | Title only | Unit Net Price (per-copy) | None |
| B — Academic eBook Catalogue | Large vendor catalogue of candidate acquisitions (1,743 records) | Title + 4-level Discipline hierarchy | None | Discipline (Level 1–4) |
| C — eBook Acquisition Catalogue | Licensing catalogue of candidate acquisitions (807 records) | Title + Category + Discipline | List price + 18 concurrent-user/term license prices | Category, Discipline |

Because the three datasets do not share a common schema, each is queried with its **own** field configuration (`predicate_engine.DATASET_FIELDS`) instead of being forced into one shape — exactly as the brief allows ("datasets may be processed separately").

A key structural fact used throughout the analysis: **Dataset A has no discipline/subject column**, so its predicate query can only search the Title — this is the main reason it returns far fewer matches than B or C in both scenarios below.


In [2]:
AFF = build_affordability_thresholds(df_a, df_c)
print("Data-driven affordability thresholds (25th/90th percentile of each dataset's own price field):")
for k, v in AFF.items():
    print(f"  Dataset {k}: field='{v['field']}'  low=RM/USD {v['low']:.2f}  high={v['high']:.2f}")


Data-driven affordability thresholds (25th/90th percentile of each dataset's own price field):
  Dataset A: field='Unit Net Price'  low=RM/USD 212.86  high=638.65
  Dataset C: field='Single user / 1-Year'  low=RM/USD 86.69  high=196.44


## 2. Predicate Query Design

A predicate query returns TRUE for a record iff any of its searchable fields contains any keyword in the scenario's keyword set (whole-word, case-insensitive). This is implemented once in `predicate_query()` and reused for both scenarios and all three datasets — the only things that change per call are the dataset key and the keyword list.

## 3. Fuzzy Reasoning Design

Three fuzzy variables are defined on filtered (predicate-satisfying) records only:

1. **Topic relevance** — `Directly Related` → 1.0, `Programming/Mathematical Support` → 0.7, `Other Justified Match` (matched only via discipline/category, not title) → 0.45.
2. **Recency** — piecewise-linear membership over age = 2026 − copyright year: ≤2y → 1.0, 2–5y → 1.0→0.6, 5–10y → 0.6→0.3, >10y → 0.2.
3. **Affordability** — piecewise-linear membership over each dataset's OWN price field, using that dataset's 25th/90th percentile as the low/high anchor (shown above). **Not evaluated for Dataset B, which has no price field at all.**

Aggregation is a weighted average (relevance 0.5 : recency 0.3 : affordability 0.2), **re-normalised over whichever components are actually available** for that record — see `fuzzy_engine.aggregate_fuzzy_score()`. This means a missing price field does not get silently defaulted to a fake neutral score; the weight is honestly redistributed to the evidence that *does* exist, and this redistribution is itself discussed in Section 7 (how available evidence changes the outcome).


In [3]:
def year_col(key):
    return {"A": "Copyright Year", "B": "Copyright", "C": "Copyright Year"}[key]

def run_query(key, keywords, direct_kw, support_map):
    """Runs predicate query then fuzzy-enrichment for one dataset."""
    pred = predicate_query(DFS[key], key, keywords)
    if len(pred) == 0:
        return pred, pred
    pred = pred.copy()
    pred['Relevance_Label'] = pred['Title'].apply(lambda t: classify_relevance(t, direct_kw, support_map))
    pred['Relevance_Score'] = pred['Relevance_Label'].apply(relevance_membership)
    ycol = year_col(key)
    pred['Recency_Score'] = pred[ycol].apply(recency_membership) if ycol in pred.columns else None
    if key in AFF:
        cfg = AFF[key]
        pred['Affordability_Score'] = pred[cfg['field']].apply(lambda p: affordability_membership(p, cfg['low'], cfg['high']))
    else:
        pred['Affordability_Score'] = None

    scores = []
    for _, row in pred.iterrows():
        comps = {"relevance": row['Relevance_Score'], "recency": row['Recency_Score'], "affordability": row['Affordability_Score']}
        s, _, _ = aggregate_fuzzy_score(comps)
        scores.append(s)
    pred['Fuzzy_Score'] = scores
    fuzzy = pred.sort_values('Fuzzy_Score', ascending=False)
    return pred, fuzzy

DISPLAY_COLS = ['Title', 'Relevance_Label', 'Recency_Score', 'Affordability_Score', 'Fuzzy_Score']


## 4. Fixed Scenario 1 — Artificial Intelligence, Programming and Mathematical Foundations

Direct AI keywords are kept separate from programming/mathematics support keywords so each returned record can be labelled by its relationship to the topic, as the brief requires.


In [4]:
direct_kw_s1 = ['Artificial Intelligence', 'Intelligent Systems', 'Machine Learning',
                'Computer Vision', 'Robotics', 'Expert Systems', 'Knowledge Representation']
prog_kw_s1 = ['Python', 'Java', 'C++', 'Algorithms', 'Data Structures']
math_kw_s1 = ['Statistics', 'Probability', 'Linear Algebra', 'Discrete Mathematics',
              'Calculus', 'Optimization', 'Decision Analysis']
s1_keywords = direct_kw_s1 + prog_kw_s1 + math_kw_s1
support_map_s1 = {"Programming Support": prog_kw_s1, "Mathematical Support": math_kw_s1}

s1_results = {}
for key in ["A", "B", "C"]:
    pred, fuzzy = run_query(key, s1_keywords, direct_kw_s1, support_map_s1)
    s1_results[key] = (pred, fuzzy)
    print(f"Dataset {key}: predicate-only matches = {len(pred)}")


Dataset A: predicate-only matches = 0
Dataset B: predicate-only matches = 194
Dataset C: predicate-only matches = 73


### Dataset A — Scenario 1

In [5]:
pred, fuzzy = s1_results['A']
if len(pred) == 0:
    print("No Dataset A records satisfy the Scenario 1 predicate.")
    print("Reason: Dataset A (9 records, current DCS-recommended holdings) contains only")
    print("Electrical/Electronics engineering (DEE), one software-engineering title, one")
    print("security title and two business/accounting titles — none contain an AI, ML,")
    print("programming-language or mathematics-foundation keyword in the Title (the only")
    print("field Dataset A exposes for search). This is a genuine structural finding, not a")
    print("bug: Dataset A currently offers no direct or supporting coverage for this scenario.")
else:
    display(fuzzy[DISPLAY_COLS].head(5))


No Dataset A records satisfy the Scenario 1 predicate.
Reason: Dataset A (9 records, current DCS-recommended holdings) contains only
Electrical/Electronics engineering (DEE), one software-engineering title, one
security title and two business/accounting titles — none contain an AI, ML,
programming-language or mathematics-foundation keyword in the Title (the only
field Dataset A exposes for search). This is a genuine structural finding, not a
bug: Dataset A currently offers no direct or supporting coverage for this scenario.


### Dataset B — Scenario 1 (predicate-only, top 5 by catalogue order, vs. fuzzy-enhanced, top 5 by score)

In [6]:
pred, fuzzy = s1_results['B']
print("Predicate-only (first 5 matches, catalogue order — no ranking notion exists yet):")
display(pred[['Title','Relevance_Label']].head(5))
print("\nFuzzy-enhanced (ranked by Fuzzy_Score):")
display(fuzzy[DISPLAY_COLS].head(5))


Predicate-only (first 5 matches, catalogue order — no ranking notion exists yet):


,Title,Relevance_Label
3,"Foundations of Decision Analysis, Global Edition",Mathematical Support
11,"Statistical Methods for the Social Sciences, Global Edition",Other Justified Match
12,"Statistics: The Art and Science of Learning from Data, Global Edition",Mathematical Support
13,"Statistical Methods for the Social Sciences, Global Edition",Other Justified Match
27,"Absolute Java, Global Edition",Programming Support



Fuzzy-enhanced (ranked by Fuzzy_Score):


,Title,Relevance_Label,Recency_Score,Affordability_Score,Fuzzy_Score
117,Artificial Intelligence: A Guide to Intelligent Systems,Directly Related,1.000000,None,1.0000
134,"Artificial Intelligence: A Modern Approach, Global Edition\n",Directly Related,0.733333,None,0.9000
129,"Artificial Intelligence: A Modern Approach, Global Edition",Directly Related,0.600000,None,0.8500
350,"Introduction to Robotics, Global Edition",Directly Related,0.600000,None,0.8500
1412,"Systems for Analytics, Data Science, & Artificial Intelligence: Systems for Decision S...",Directly Related,0.540000,None,0.8275


### Dataset C — Scenario 1

In [7]:
pred, fuzzy = s1_results['C']
print("Predicate-only (first 5 matches, catalogue order):")
display(pred[['Title','Relevance_Label']].head(5))
print("\nFuzzy-enhanced (ranked by Fuzzy_Score):")
display(fuzzy[DISPLAY_COLS].head(5))


Predicate-only (first 5 matches, catalogue order):


,Title,Relevance_Label
10,Introduction Techinical Mathematics,Other Justified Match
28,Calculus,Mathematical Support
29,"Calculus, Metric Edition",Mathematical Support
30,Calculus: Early Transcendentals,Mathematical Support
31,"Calculus: Early Transcendentals, Metric Edition",Mathematical Support



Fuzzy-enhanced (ranked by Fuzzy_Score):


,Title,Relevance_Label,Recency_Score,Affordability_Score,Fuzzy_Score
163,"Artificial Intelligence, 2e",Directly Related,0.733333,1.000000,0.9200
421,Introduction to Artificial Intelligence: A Business Perspective,Directly Related,1.000000,0.547546,0.9095
777,"Artificial Intelligence, Analytics and Data Science (Vol. 1)",Directly Related,0.600000,1.000000,0.8800
155,Android Boot Camp for Developers Using Java®,Programming Support,1.000000,0.930431,0.8361
534,"Business Statistics: Using Excel, SPSS, and R",Mathematical Support,0.866667,1.000000,0.8100


**Scenario 1 comparison (predicate-only vs fuzzy-enhanced):** In predicate-only mode, all 194 (B) / 73 (C) matches are treated as equally satisfying — the department would have to read every title. Fuzzy enhancement reorders them so `Directly Related` AI titles with recent copyright years surface first, and (for Dataset C only) cheaper licences are preferred among otherwise-similar records. Dataset B's ranking is driven almost entirely by relevance + recency, since affordability evidence does not exist for it — the 0.5/0.3/0.2 weights are re-normalised to ≈0.625/0.375 automatically.


## 5. Fixed Scenario 2 — Cybersecurity and Secure Computing

Per the brief, **all** relevant Dataset A records are shown in full because Dataset A represents the department's **current subscription** (its existing collection) — knowing exactly what is already held is more important here than capping the list. Datasets B and C (candidate *new* acquisitions) are capped at 10 records as instructed.


In [8]:
s2_keywords = ['Cybersecurity', 'Computer Security', 'Network Security', 'Cryptography',
               'Privacy', 'Digital Forensics', 'Information Assurance', 'Secure Systems', 'Security']
# Scenario 2 has no separate "support" sub-topic in the brief; all listed areas count as direct relevance
support_map_s2 = {}

s2_results = {}
for key in ["A", "B", "C"]:
    pred, fuzzy = run_query(key, s2_keywords, s2_keywords, support_map_s2)
    s2_results[key] = (pred, fuzzy)
    print(f"Dataset {key}: predicate-only matches = {len(pred)}")


Dataset A: predicate-only matches = 1
Dataset B: predicate-only matches = 9
Dataset C: predicate-only matches = 6


### Dataset A — Scenario 2 (Current Subscription: show ALL matches)

In [9]:
pred, fuzzy = s2_results['A']
display(fuzzy[DISPLAY_COLS])


,Title,Relevance_Label,Recency_Score,Affordability_Score,Fuzzy_Score
5,Security in Computing,Directly Related,1.0,0.1,0.82


### Dataset B — Scenario 2 (candidate acquisitions, up to 10)

In [10]:
pred, fuzzy = s2_results['B']
print("Predicate-only (first matches, catalogue order):")
display(pred[['Title','Relevance_Label']].head(10))
print("\nFuzzy-enhanced (ranked):")
display(fuzzy[DISPLAY_COLS].head(10))


Predicate-only (first matches, catalogue order):


,Title,Relevance_Label
200,"Boyle: Corporate Computer Security, Global Edition",Directly Related
421,"Computer Security: Principles and Practice, Global Edition",Directly Related
422,"Computer Security: Principles and Practice, Global Edition",Directly Related
430,"Cryptography and Network Security: Principles and Practice, Global Edition",Directly Related
848,Introduction to Computer Security,Directly Related
1228,"Business Data Networks and Security, Global Edition",Directly Related
1308,"Network Security Essentials: Applications and Standards, Global Edition",Directly Related
1389,Practical Cryptology and Web Security,Directly Related
1741,Security in Computing,Directly Related



Fuzzy-enhanced (ranked):


,Title,Relevance_Label,Recency_Score,Affordability_Score,Fuzzy_Score
421,"Computer Security: Principles and Practice, Global Edition",Directly Related,1.000000,None,1.0000
1741,Security in Computing,Directly Related,1.000000,None,1.0000
430,"Cryptography and Network Security: Principles and Practice, Global Edition",Directly Related,0.733333,None,0.9000
1308,"Network Security Essentials: Applications and Standards, Global Edition",Directly Related,0.480000,None,0.8050
422,"Computer Security: Principles and Practice, Global Edition",Directly Related,0.420000,None,0.7825
200,"Boyle: Corporate Computer Security, Global Edition",Directly Related,0.200000,None,0.7000
848,Introduction to Computer Security,Directly Related,0.200000,None,0.7000
1228,"Business Data Networks and Security, Global Edition",Directly Related,0.200000,None,0.7000
1389,Practical Cryptology and Web Security,Directly Related,0.200000,None,0.7000


### Dataset C — Scenario 2 (candidate acquisitions, up to 10; price field = 'Single user / 1-Year' licence)

In [11]:
pred, fuzzy = s2_results['C']
print("Predicate-only (first matches, catalogue order):")
display(pred[['Title','Relevance_Label']].head(10))
print("\nFuzzy-enhanced (ranked):")
display(fuzzy[DISPLAY_COLS].head(10))


Predicate-only (first matches, catalogue order):


,Title,Relevance_Label
228,CompTIA CySA+ Guide to Cybersecurity Analyst (CS0-003),Directly Related
231,CompTIA Security+ Guide to Network Security Fundamentals,Directly Related
441,Management of Cybersecurity,Directly Related
542,Chinese Rice Bowl: Understanding Food Security in China,Directly Related
667,Principles of Information Security,Directly Related
677,Security Awareness: Applying Practical Cybersecurity in Your World,Directly Related



Fuzzy-enhanced (ranked):


,Title,Relevance_Label,Recency_Score,Affordability_Score,Fuzzy_Score
677,Security Awareness: Applying Practical Cybersecurity in Your World,Directly Related,1.000000,1.000000,1.0000
542,Chinese Rice Bowl: Understanding Food Security in China,Directly Related,0.866667,1.000000,0.9600
441,Management of Cybersecurity,Directly Related,1.000000,0.746436,0.9493
228,CompTIA CySA+ Guide to Cybersecurity Analyst (CS0-003),Directly Related,1.000000,0.418224,0.8836
231,CompTIA Security+ Guide to Network Security Fundamentals,Directly Related,1.000000,0.418224,0.8836
667,Principles of Information Security,Directly Related,0.733333,0.418224,0.8036


**Important limitation surfaced by real data:** Dataset C's predicate match list includes *"Chinese Rice Bowl: Understanding Food Security in China"*. It satisfies the crisp predicate because it contains the word **"Security"**, but it is about food security, not cybersecurity — a **false positive of keyword-only predicate matching**. Fuzzy scoring does not fix this (it still receives a `Directly Related` relevance label from the same keyword match); only a smarter predicate (e.g. requiring co-occurrence with "computer/cyber/network/information") or a human review step would catch it. This is used directly in Section 7 (Critical Analysis) as evidence of a genuine predicate-only weakness observed in this project, not a hypothetical one.


## 6. Comparison and Analysis Across Both Scenarios

| | Dataset A (Existing) | Dataset B (Academic Catalogue) | Dataset C (Acquisition Catalogue) |
|---|---|---|---|
| Size | 9 | 1,743 | 807 |
| Discipline/subject field? | No | Yes (4 levels) | Yes (Category + Discipline) |
| Price field? | Yes (Unit Net Price) | **No** | Yes (list price + 18 licence tiers) |
| Scenario 1 predicate matches | 0 | 194 | 73 |
| Scenario 2 predicate matches | 1 | 9 | 6 |


In [12]:
summary_rows = []
for scen_label, res in [("S1", s1_results), ("S2", s2_results)]:
    for key in ["A","B","C"]:
        pred, fuzzy = res[key]
        summary_rows.append({
            "Scenario": scen_label, "Dataset": key, "Predicate Matches": len(pred),
            "Has Affordability Evidence": key in AFF,
            "Top Fuzzy Score": (fuzzy['Fuzzy_Score'].max() if len(fuzzy) else None)
        })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


,Scenario,Dataset,Predicate Matches,Has Affordability Evidence,Top Fuzzy Score
0,S1,A,0,True,NaN
1,S1,B,194,False,1.00
2,S1,C,73,True,0.92
3,S2,A,1,True,0.82
4,S2,B,9,False,1.00
5,S2,C,6,True,1.00


**How dataset size, structure and available evidence affected the outcome:**

- **Size dominates recall.** Dataset A (9 records) can structurally never return many matches — it returned 0/9 for Scenario 1 and 1/9 for Scenario 2, regardless of how good the predicate or fuzzy logic is. Dataset B, the largest (1,743), returned the most matches in both scenarios.
- **Discipline metadata improves precision AND enables a second search path.** Datasets B and C can be matched via their discipline/category hierarchy as well as the title, so a book like a general "Machine Learning" text that only mentions the topic in its subject classification (not its literal title) is still found — impossible in Dataset A, which has no such field.
- **Missing price evidence changes what fuzzy reasoning *can* claim.** Dataset B's fuzzy score is entirely relevance+recency (weights re-normalised ≈0.625/0.375); Datasets A and C additionally fold in affordability. This means Dataset B rankings should be read as "most topically strong and recent," not "best value," which is an important caveat for the department.
- **Keyword predicate breadth trades recall for precision.** A broad single-word keyword like "Security" maximises recall (catches every genuinely relevant title) but also lets through false positives like the food-security book in Dataset C — visible directly in the Scenario 2 output above.
- **Fuzzy reasoning adds the most value exactly where predicate-only leaves the department with a long, unordered list** (B: 194/9 matches, C: 73/6 matches) and adds the least value where predicate-only already returns 0–1 record (Dataset A), since there is nothing left to rank.


## 7. Conclusion

**Predicate-only strengths:** transparent, deterministic, cheap to compute, and sufficient when the candidate set is already tiny (Dataset A).
**Predicate-only limitations:** binary in/out decision hides genuinely useful differences in quality (recency, price, depth of relevance), and simple keyword predicates are vulnerable to false positives (the food-security example).

**Fuzzy-enhanced strengths:** turns a long undifferentiated candidate list (Dataset B/C) into an actionable, ranked shortlist, and makes trade-offs (e.g. relevance vs. price) explicit and auditable via the aggregation weights.
**Fuzzy-enhanced limitations:** the membership functions and weights are still human-designed judgement calls (not learned from department feedback), and fuzzy reasoning cannot correct a predicate-level false positive — it can only re-order what predicate filtering already let through.

**Improvement directions:** (1) tighten Scenario 2's predicate with co-occurrence rules to reduce false positives like the food-security case; (2) add a discipline-match-strength fuzzy variable for Dataset A if a subject-tagging exercise is ever done on the existing collection; (3) elicit real weight preferences from the DCS department rather than assuming 0.5/0.3/0.2.

## References

Dubois, D., & Prade, H. (1996). What are fuzzy rules and how to use them. *Fuzzy Sets and Systems, 84*(2), 169–185. https://doi.org/10.1016/0165-0114(96)00066-8

Russell, S., & Norvig, P. (2021). *Artificial intelligence: A modern approach* (4th ed.). Pearson.
